In [ ]:
import pandas as pd

# Leer el archivo parquet
df = pd.read_parquet("NHANES.parquet")

In [ ]:
print(df.dtypes)
print(df.select_dtypes(include='category').columns)

In [ ]:
# Porcentaje de NA por variable
na_percent = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(na_percent[na_percent > 0])

In [ ]:
# Información básica del dataset
print("=== INFORMACIÓN BÁSICA ===")
print(f"Dimensiones: {df.shape}")
print(f"Tipos de datos:\n{df.dtypes.value_counts()}")

# Lista de todas las variables con sus tipos
print("\n=== VARIABLES Y TIPOS ===")
var_info = pd.DataFrame({
    'Variable': df.columns,
    'Tipo': df.dtypes,
    'NA_count': df.isnull().sum(),
    'NA_percent': (df.isnull().sum() / len(df) * 100).round(2)
})

print(var_info.to_string(index=False))

# Para copiar fácilmente - lista de variables categóricas
print("\n=== VARIABLES CATEGÓRICAS ===")
categorical_vars = df.select_dtypes(include='category').columns.tolist()
print(categorical_vars)

# Para copiar fácilmente - lista de variables numéricas
print("\n=== VARIABLES NUMÉRICAS ===")
numeric_vars = df.select_dtypes(include=['int32', 'int64', 'float64']).columns.tolist()
print(numeric_vars)

In [ ]:
# Lista de variables a eliminar (25 variables)
variables_eliminar = [
   # >90% de valores faltantes
   'HeadCirc', 'Length', 'DiabetesAge', 'TVHrsDayChild', 'CompHrsDayChild',

   # Variables específicas de subgrupos (80-90% NA)
   'BMICatUnder20yrs', 'AgeRegMarij', 'UrineVol2', 'UrineFlow2',
   'PregnantNow', 'Age1stBaby',

   # Variables demográficamente específicas (70-80% NA)
   'nBabies', 'nPregnancies', 'AgeFirstMarij', 'SmokeAge',

   # Variables sensibles con alta proporción de NA (>40%)
   'Testosterone', 'SexOrientation', 'SexNumPartYear', 'Race3', 'AgeMonths',
   'AlcoholDay', 'SexAge', 'SexNumPartnLife', 'HardDrugs', 'SexEver', 'SameSex',

   # Variable ID
   'ID'
]

print(f"Variables a eliminar: {len(variables_eliminar)}")

# Verificar que todas las variables existen en el dataset
variables_no_encontradas = [var for var in variables_eliminar if var not in df.columns]
if variables_no_encontradas:
   print(f"ADVERTENCIA: Variables no encontradas: {variables_no_encontradas}")

# Eliminar las variables
df_clean = df.drop(columns=variables_eliminar, errors='ignore')

print(f"Dataset limpio: {df_clean.shape}")
print(f"Variables eliminadas: {df.shape[1] - df_clean.shape[1]}")

# Verificar el nuevo análisis de NA
print("\n=== ANÁLISIS DE NA EN DATASET LIMPIO ===")
na_percent_clean = (df_clean.isnull().sum() / len(df_clean) * 100).sort_values(ascending=False)
print("Variables con NA (ordenadas por %):")
print(na_percent_clean[na_percent_clean > 0].round(2))

# Clasificar variables por umbrales de NA
print("\n=== CLASIFICACIÓN POR UMBRALES ===")
vars_careful = na_percent_clean[(na_percent_clean >= 50) & (na_percent_clean <= 80)]
vars_impute = na_percent_clean[(na_percent_clean >= 10) & (na_percent_clean < 50)]
vars_safe = na_percent_clean[(na_percent_clean > 0) & (na_percent_clean < 10)]
vars_complete = na_percent_clean[na_percent_clean == 0]

print(f"Variables 50-80% NA (evaluar cuidadosamente): {len(vars_careful)}")
if len(vars_careful) > 0:
   print(f"  {vars_careful.index.tolist()}")

print(f"Variables 10-50% NA (candidatas a imputación): {len(vars_impute)}")
if len(vars_impute) > 0:
   print(f"  {vars_impute.index.tolist()}")

print(f"Variables <10% NA (seguras): {len(vars_safe)}")
if len(vars_safe) > 0:
   print(f"  {vars_safe.index.tolist()}")

print(f"Variables completas (0% NA): {len(vars_complete)}")

# Guardar el dataset limpio
df_clean.to_parquet('nhanes_limpieza_1.parquet', index=False)
print(f"\nDataset limpio guardado como 'nhanes_limpieza_1.parquet'")

# Mostrar las variables restantes por tipo
print(f"\n=== VARIABLES RESTANTES POR TIPO ===")
print(f"Categóricas: {len(df_clean.select_dtypes(include='category').columns)}")
print(f"Numéricas: {len(df_clean.select_dtypes(include=['int32', 'int64', 'float64']).columns)}")

In [ ]:
import pandas as pd

# Cargar dataset limpio
df_clean = pd.read_parquet('nhanes_limpieza_1.parquet')

print("=== ANÁLISIS PARA SEGUNDA LIMPIEZA ===")

# Variables con 50-80% NA para evaluar individualmente
vars_50_80 = ['SmokeNow', 'PhysActiveDays', 'TVHrsDay', 'CompHrsDay', 'Marijuana', 'RegularMarij']

print("\n1. VARIABLES 50-80% NA A EVALUAR:")
for var in vars_50_80:
    na_pct = (df_clean[var].isnull().sum() / len(df_clean) * 100)
    print(f"  {var}: {na_pct:.1f}% NA")
    if var in df_clean.select_dtypes(include='category').columns:
        print(f"    Categorías: {df_clean[var].cat.categories.tolist()}")
    print()

# Buscar variables potencialmente redundantes
print("2. VARIABLES POTENCIALMENTE REDUNDANTES:")

# Variables de presión arterial (múltiples mediciones)
bp_vars = ['BPSysAve', 'BPDiaAve', 'BPSys1', 'BPDia1', 'BPSys2', 'BPDia2', 'BPSys3', 'BPDia3']
print(f"  Presión arterial (8 vars): {bp_vars}")

# Variables de fumar
smoke_vars = ['SmokeNow', 'Smoke100', 'Smoke100n']
print(f"  Fumar (3 vars): {smoke_vars}")

# Variables de salud mental
mental_vars = ['LittleInterest', 'Depressed', 'DaysPhysHlthBad', 'DaysMentHlthBad']
print(f"  Salud mental (4 vars): {mental_vars}")

# Variables de ingresos
income_vars = ['HHIncome', 'HHIncomeMid', 'Poverty']
print(f"  Ingresos (3 vars): {income_vars}")

print("\n3. CORRELACIONES ENTRE VARIABLES NUMÉRICAS RELACIONADAS:")

# Correlaciones BP
bp_numeric = [var for var in bp_vars if var in df_clean.select_dtypes(include=['float64', 'int32']).columns]
if len(bp_numeric) > 1:
    print("Correlaciones presión arterial:")
    print(df_clean[bp_numeric].corr().round(3))

# Correlaciones ingresos
income_numeric = [var for var in income_vars if var in df_clean.select_dtypes(include=['float64', 'int32']).columns]
if len(income_numeric) > 1:
    print("\nCorrelaciones ingresos:")
    print(df_clean[income_numeric].corr().round(3))

print(f"\n4. RESUMEN ACTUAL:")
print(f"Total variables: {df_clean.shape[1]}")
print(f"Variables categóricas: {len(df_clean.select_dtypes(include='category').columns)}")
print(f"Variables numéricas: {len(df_clean.select_dtypes(include=['float64', 'int32']).columns)}")

In [ ]:
# Cargar dataset de la primera limpieza
df_clean = pd.read_parquet('nhanes_limpieza_1.parquet')

print(f"Dataset antes de segunda limpieza: {df_clean.shape}")

# Variables a eliminar en la segunda limpieza (12 variables)
segunda_eliminacion = [
    # Presión arterial redundantes (6 variables)
    'BPSys1', 'BPDia1', 'BPSys2', 'BPDia2', 'BPSys3', 'BPDia3',

    # Variables de fumar redundantes/problemáticas (2 variables)
    'SmokeNow', 'Smoke100n',

    # Ingresos redundante (1 variable)
    'HHIncomeMid',

    # Variables con >50% NA poco informativas (3 variables)
    'PhysActiveDays', 'TVHrsDay', 'CompHrsDay'
]

print(f"Variables a eliminar en segunda limpieza: {len(segunda_eliminacion)}")

# Eliminar las variables
df_final = df_clean.drop(columns=segunda_eliminacion, errors='ignore')

print(f"Dataset final: {df_final.shape}")
print(f"Variables eliminadas en segunda limpieza: {df_clean.shape[1] - df_final.shape[1]}")

# Análisis final de NA
print("\n=== ANÁLISIS FINAL DE NA ===")
na_percent_final = (df_final.isnull().sum() / len(df_final) * 100).sort_values(ascending=False)
print("Variables con NA (ordenadas por %):")
print(na_percent_final[na_percent_final > 0].round(2))

# Clasificación final por umbrales
print("\n=== CLASIFICACIÓN FINAL ===")
vars_impute_final = na_percent_final[(na_percent_final >= 10) & (na_percent_final < 50)]
vars_safe_final = na_percent_final[(na_percent_final > 0) & (na_percent_final < 10)]
vars_complete_final = na_percent_final[na_percent_final == 0]

print(f"Variables 10-50% NA (imputación): {len(vars_impute_final)}")
print(f"Variables <10% NA (seguras): {len(vars_safe_final)}")
print(f"Variables completas: {len(vars_complete_final)}")

# Resumen por tipo
print(f"\n=== DATASET FINAL ===")
print(f"Total variables: {df_final.shape[1]}")
print(f"Categóricas: {len(df_final.select_dtypes(include='category').columns)}")
print(f"Numéricas: {len(df_final.select_dtypes(include=['float64', 'int32']).columns)}")

# Guardar dataset final
df_final.to_parquet('nhanes_limpieza_2.parquet', index=False)
print(f"\nDataset guardado como 'nhanes_limpieza_2.parquet'")

# Mostrar resumen de eliminaciones totales
print(f"\n=== RESUMEN TOTAL DE ELIMINACIONES ===")
print(f"Variables originales: 76")
print(f"Primera eliminación: 27 variables")
print(f"Segunda eliminación: 12 variables")
print(f"Total eliminado: 39 variables")
print(f"Variables finales: 37 variables")

In [ ]:
import pandas as pd
import numpy as np

# Cargar dataset final
df_final = pd.read_parquet('nhanes_limpieza_2.parquet')

print("=== ANÁLISIS DETALLADO DE LAS 37 VARIABLES FINALES ===")

# 1. Lista completa de variables restantes
print(f"Variables finales ({df_final.shape[1]}):")
print(df_final.columns.tolist())

# 2. Análisis detallado de NA
na_analysis = pd.DataFrame({
    'Variable': df_final.columns,
    'Tipo': df_final.dtypes,
    'NA_count': df_final.isnull().sum(),
    'NA_percent': (df_final.isnull().sum() / len(df_final) * 100).round(2),
    'Valores_unicos': [df_final[col].nunique() for col in df_final.columns]
})

na_analysis = na_analysis.sort_values('NA_percent', ascending=False)
print(f"\n=== ANÁLISIS COMPLETO POR VARIABLE ===")
print(na_analysis.to_string(index=False))

# 3. Variables categóricas con pocas categorías (posibles candidatas)
print(f"\n=== VARIABLES CATEGÓRICAS CON POCAS CATEGORÍAS ===")
cat_vars = df_final.select_dtypes(include='category').columns
for var in cat_vars:
    n_cats = df_final[var].nunique()
    na_pct = (df_final[var].isnull().sum() / len(df_final) * 100)
    print(f"{var}: {n_cats} categorías, {na_pct:.1f}% NA")
    if na_pct < 50:  # Solo mostrar categorías si no tiene demasiados NA
        print(f"  Categorías: {df_final[var].cat.categories.tolist()}")
    print()

# 4. Correlaciones entre variables numéricas restantes
print(f"=== CORRELACIONES ALTAS ENTRE VARIABLES NUMÉRICAS ===")
numeric_vars = df_final.select_dtypes(include=['float64', 'int32']).columns
if len(numeric_vars) > 1:
    corr_matrix = df_final[numeric_vars].corr()

    # Encontrar correlaciones altas (>0.8) excluyendo diagonal
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) > 0.8 and not np.isnan(corr_val):
                var1 = corr_matrix.columns[i]
                var2 = corr_matrix.columns[j]
                high_corr_pairs.append((var1, var2, corr_val))

    if high_corr_pairs:
        print("Pares con correlación >0.8:")
        for var1, var2, corr in high_corr_pairs:
            print(f"  {var1} - {var2}: {corr:.3f}")
    else:
        print("No hay correlaciones >0.8 entre variables numéricas")

# 5. Variables con muy poca variabilidad
print(f"\n=== VARIABLES CON POCA VARIABILIDAD ===")
for var in df_final.columns:
    unique_vals = df_final[var].nunique()
    total_vals = len(df_final) - df_final[var].isnull().sum()
    if unique_vals <= 3 and total_vals > 100:  # Pocas categorías pero suficientes datos
        na_pct = (df_final[var].isnull().sum() / len(df_final) * 100)
        print(f"{var}: {unique_vals} valores únicos, {na_pct:.1f}% NA")
        print(f"  Distribución: {df_final[var].value_counts().to_dict()}")
        print()

# 6. Resumen para decisión
print(f"=== RESUMEN PARA DECISIÓN FINAL ===")
print(f"Variables con >40% NA: {len(na_analysis[na_analysis['NA_percent'] > 40])}")
print(f"Variables con 20-40% NA: {len(na_analysis[(na_analysis['NA_percent'] >= 20) & (na_analysis['NA_percent'] <= 40)])}")
print(f"Variables con <20% NA: {len(na_analysis[na_analysis['NA_percent'] < 20])}")

In [ ]:
# Cargar dataset
df_final = pd.read_parquet('nhanes_limpieza_2.parquet')

print("=== ANÁLISIS ESPECÍFICO DE HomeOwn Y COLESTEROL ===")

# 1. ANÁLISIS DE HomeOwn vs otras variables sociodemográficas
print("1. CORRELACIÓN HomeOwn CON VARIABLES SOCIODEMOGRÁFICAS:")

# HomeOwn vs HHIncome (ambas categóricas)
print("\nHomeOwn vs HHIncome (crosstab):")
crosstab_income = pd.crosstab(df_final['HomeOwn'], df_final['HHIncome'],
                              normalize='index', dropna=False) * 100
print(crosstab_income.round(1))

# HomeOwn vs Education
print("\nHomeOwn vs Education (crosstab):")
crosstab_edu = pd.crosstab(df_final['HomeOwn'], df_final['Education'],
                           normalize='index', dropna=False) * 100
print(crosstab_edu.round(1))

# HomeOwn vs Poverty (numérica)
print(f"\nHomeOwn vs Poverty (promedios):")
poverty_by_home = df_final.groupby('HomeOwn', observed=False)['Poverty'].agg(['mean', 'median', 'count'])
print(poverty_by_home.round(2))

# 2. ANÁLISIS DE COLESTEROL
print(f"\n2. CORRELACIÓN ENTRE DirectChol Y TotChol:")

# Filtrar casos donde ambos tienen datos
both_chol = df_final[['DirectChol', 'TotChol']].dropna()
print(f"Casos con ambos valores: {len(both_chol)} de {len(df_final)} ({len(both_chol)/len(df_final)*100:.1f}%)")

if len(both_chol) > 100:  # Solo si hay suficientes casos
    correlation = both_chol['DirectChol'].corr(both_chol['TotChol'])
    print(f"Correlación DirectChol - TotChol: {correlation:.3f}")

    # Estadísticas descriptivas
    print(f"\nEstadísticas DirectChol:")
    print(both_chol['DirectChol'].describe().round(2))

    print(f"\nEstadísticas TotChol:")
    print(both_chol['TotChol'].describe().round(2))

    # Ver diferencias
    print(f"\nRango de valores:")
    print(f"DirectChol: {both_chol['DirectChol'].min():.1f} - {both_chol['DirectChol'].max():.1f}")
    print(f"TotChol: {both_chol['TotChol'].min():.1f} - {both_chol['TotChol'].max():.1f}")

else:
    print("Insuficientes casos para análisis de correlación")

# 3. ANÁLISIS DE PATRONES DE NA
print(f"\n3. PATRONES DE NA:")
print(f"DirectChol NA: {df_final['DirectChol'].isnull().sum()} ({df_final['DirectChol'].isnull().mean()*100:.1f}%)")
print(f"TotChol NA: {df_final['TotChol'].isnull().sum()} ({df_final['TotChol'].isnull().mean()*100:.1f}%)")

# Ver si los NA coinciden
both_na = df_final['DirectChol'].isnull() & df_final['TotChol'].isnull()
print(f"Ambos NA simultáneamente: {both_na.sum()} casos")

# Ver casos donde uno tiene dato y otro no
direct_only = (~df_final['DirectChol'].isnull()) & (df_final['TotChol'].isnull())
total_only = (df_final['DirectChol'].isnull()) & (~df_final['TotChol'].isnull())
print(f"Solo DirectChol disponible: {direct_only.sum()} casos")
print(f"Solo TotChol disponible: {total_only.sum()} casos")

In [ ]:
import pandas as pd

# Cargar dataset de la segunda limpieza
df_final = pd.read_parquet('nhanes_limpieza_2.parquet')

print(f"Dataset antes de limpieza final: {df_final.shape}")

# Variables a eliminar en la limpieza final (3 variables)
eliminacion_final = [
    'RegularMarij',    # Redundante con Marijuana, mismo % NA (50.59%)
    'Weight',          # Correlación 0.902 con BMI
    'SurveyYr'         # Variable administrativa sin valor predictivo
]

print(f"Variables a eliminar: {eliminacion_final}")

# Eliminar las variables
df_definitivo = df_final.drop(columns=eliminacion_final, errors='ignore')

# Eliminar filas con Diabetes sin informar
df_definitivo = df_definitivo.dropna(subset=["Diabetes"])

print(f"Dataset definitivo (última elminación y eliminando filas sin Diabetes informado): {df_definitivo.shape}")
print(f"Variables eliminadas: {df_final.shape[1] - df_definitivo.shape[1]}")

# Análisis final completo
print("\n=== ANÁLISIS FINAL DEL DATASET DEFINITIVO ===")
na_percent_definitivo = (df_definitivo.isnull().sum() / len(df_definitivo) * 100).sort_values(ascending=False)

print("Variables con NA (ordenadas por %):")
print(na_percent_definitivo[na_percent_definitivo > 0].round(2))

# Clasificación final por umbrales
print("\n=== CLASIFICACIÓN FINAL POR UMBRALES ===")
vars_impute_definitivo = na_percent_definitivo[(na_percent_definitivo >= 10) & (na_percent_definitivo < 50)]
vars_safe_definitivo = na_percent_definitivo[(na_percent_definitivo > 0) & (na_percent_definitivo < 10)]
vars_complete_definitivo = na_percent_definitivo[na_percent_definitivo == 0]

print(f"Variables 10-50% NA (imputación): {len(vars_impute_definitivo)}")
print(f"  {vars_impute_definitivo.index.tolist()}")

print(f"Variables <10% NA (seguras): {len(vars_safe_definitivo)}")
print(f"  {vars_safe_definitivo.index.tolist()}")

print(f"Variables completas (0% NA): {len(vars_complete_definitivo)}")
print(f"  {vars_complete_definitivo.index.tolist()}")

# Resumen por tipo
print(f"\n=== ESTRUCTURA FINAL ===")
print(f"Total variables: {df_definitivo.shape[1]}")
categoricas = df_definitivo.select_dtypes(include='category').columns
numericas = df_definitivo.select_dtypes(include=['float64', 'int32']).columns
print(f"Variables categóricas: {len(categoricas)}")
print(f"Variables numéricas: {len(numericas)}")

# Guardar dataset definitivo
df_definitivo.to_parquet('nhanes_limpio.parquet', index=False)
print(f"\n✅ Dataset definitivo guardado como 'nhanes_limpio.parquet'")

# Resumen total de todo el proceso
print(f"\n=== RESUMEN TOTAL DEL PROCESO DE LIMPIEZA ===")
print(f"📊 Variables originales: 76")
print(f"❌ Primera eliminación: 27 variables")
print(f"❌ Segunda eliminación: 12 variables")
print(f"❌ Tercera eliminación: 3 variables")
print(f"📈 Total eliminado: 42 variables")
print(f"✅ Variables finales: 34 variables")
print(f"📉 Reducción: {42/76*100:.1f}% del dataset original")

# Lista final de variables
print(f"\n=== VARIABLES FINALES ({df_definitivo.shape[1]}) ===")
print("Categóricas:", categoricas.tolist())
print("Numéricas:", numericas.tolist())